# Day 13 — JOINs: All Types + Diagnostics
**Estimated time:** 60–75 min
**Datasets:** `sales_orders.csv`, `materials_inventory.csv`

## Learning Objectives
- Execute all JOIN types in SQLite, including simulations for RIGHT and FULL OUTER
- Use self-joins for year-over-year comparisons
- Apply join diagnostic patterns: null detection, row count verification, duplicate analysis

In [ ]:
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path

from analytics_bootcamp.config import RAW_DATA_DIR as DATA_DIR
# Load all CSVs
inv   = pd.read_csv(f"{DATA_DIR}/materials_inventory.csv")
sales = pd.read_csv(f"{DATA_DIR}/sales_orders.csv")
cc    = pd.read_csv(f"{DATA_DIR}/cost_center_actuals.csv")
hr    = pd.read_csv(f"{DATA_DIR}/hr_headcount.csv")
bw    = pd.read_csv(f"{DATA_DIR}/bw_sales_kpis.csv")

# Normalize column names
for df in [inv, sales, cc, hr, bw]:
    df.columns = [c.strip().upper() for c in df.columns]

# In-memory DuckDB — register pandas DataFrames as tables
con = duckdb.connect()
con.register("MATERIALS",   inv)
con.register("SALES",       sales)
con.register("COST_CENTER", cc)
con.register("HR",          hr)
con.register("BW_SALES",    bw)

def q(sql):
    return con.sql(sql).df()

# Verify
for tbl, df in [("MATERIALS",inv),("SALES",sales),("COST_CENTER",cc),("HR",hr),("BW_SALES",bw)]:
    n = q(f"SELECT COUNT(*) AS n FROM {tbl}").iloc[0,0]
    print(f"  {tbl:15s}: {n:,} rows")


In [ ]:
# -- Cardinality check before any join --
result = q(
    """
    SELECT 'SALES.MATNR'     AS source, COUNT(*) AS total_rows, COUNT(DISTINCT MATNR) AS distinct_matnr FROM SALES
    UNION ALL
    SELECT 'MATERIALS.MATNR',           COUNT(*),                COUNT(DISTINCT MATNR)                  FROM MATERIALS
    """
)
display(result)

In [ ]:
# -- INNER JOIN: sales matched to material master --
result = q(
    """
    SELECT
        s.VBELN, s.MATNR, s.NETWR, s.STATUS, s.ERDAT,
        m.MAKTX, m.LABST, m.STPRS, m.MATERIAL_TYPE
    FROM SALES s
    INNER JOIN MATERIALS m ON s.MATNR = m.MATNR
    ORDER BY s.NETWR DESC
    LIMIT 10
    """
)
print(f"INNER JOIN rows: {len(result):,}")
display(result.head())

In [ ]:
# -- LEFT JOIN: keep all sales, null if no material record --
result = q(
    """
    SELECT
        s.VBELN, s.MATNR, s.NETWR,
        m.MAKTX, m.LABST,
        CASE WHEN m.MATNR IS NULL THEN 'NO MATERIAL MASTER' ELSE 'MATCHED' END AS join_status
    FROM SALES s
    LEFT JOIN MATERIALS m ON s.MATNR = m.MATNR
    """
)
print(result["JOIN_STATUS"].value_counts())
display(result[result["JOIN_STATUS"] == "NO MATERIAL MASTER"].head(10))

In [ ]:
# -- RIGHT JOIN simulation (SQLite lacks RIGHT JOIN natively) --
# Swap table order in a LEFT JOIN to get all rows from MATERIALS
result = q(
    """
    SELECT
        m.MATNR, m.MAKTX, m.LABST,
        s.VBELN, s.NETWR,
        CASE WHEN s.MATNR IS NULL THEN 'NO SALES' ELSE 'HAS SALES' END AS sales_status
    FROM MATERIALS m
    LEFT JOIN SALES s ON m.MATNR = s.MATNR
    """
)
print(result["SALES_STATUS"].value_counts())
display(result[result["SALES_STATUS"] == "NO SALES"].head(10))

In [ ]:
# -- FULL OUTER JOIN simulation: UNION of two LEFT JOINs --
result = q(
    """
    SELECT COALESCE(s.MATNR, m.MATNR) AS MATNR,
           s.VBELN, s.NETWR, m.MAKTX, m.LABST,
           CASE WHEN s.MATNR IS NULL THEN 'Materials only'
                WHEN m.MATNR IS NULL THEN 'Sales only'
                ELSE 'Both' END AS coverage
    FROM SALES s LEFT JOIN MATERIALS m ON s.MATNR = m.MATNR
    UNION
    SELECT COALESCE(s.MATNR, m.MATNR),
           s.VBELN, s.NETWR, m.MAKTX, m.LABST,
           CASE WHEN s.MATNR IS NULL THEN 'Materials only'
                WHEN m.MATNR IS NULL THEN 'Sales only'
                ELSE 'Both' END
    FROM MATERIALS m LEFT JOIN SALES s ON m.MATNR = s.MATNR
    """
)
print(result["COVERAGE"].value_counts())

In [ ]:
# -- Self-join: current year vs prior year spend per cost center --
result = q(
    """
    SELECT
        cy.KOSTL,
        cy.KTEXT,
        cy.GJAHR       AS current_year,
        cy.PERIOD,
        cy.ACTUAL_AMT  AS current_amt,
        py.ACTUAL_AMT  AS prior_amt,
        cy.ACTUAL_AMT - COALESCE(py.ACTUAL_AMT, 0) AS yoy_delta,
        ROUND(
            (cy.ACTUAL_AMT - COALESCE(py.ACTUAL_AMT, 0)) * 100.0
            / NULLIF(py.ACTUAL_AMT, 0)
        , 2) AS yoy_pct
    FROM COST_CENTER cy
    LEFT JOIN COST_CENTER py
        ON cy.KOSTL = py.KOSTL
       AND cy.PERIOD = py.PERIOD
       AND cy.GJAHR = py.GJAHR + 1
    WHERE cy.GJAHR = (SELECT MAX(GJAHR) FROM COST_CENTER)
    ORDER BY yoy_pct DESC
    LIMIT 15
    """
)
display(result)

In [ ]:
# -- Join diagnostic: detect many-to-many duplicates --
result = q(
    """
    SELECT s.VBELN, s.MATNR, COUNT(*) AS join_row_count
    FROM SALES s
    INNER JOIN MATERIALS m ON s.MATNR = m.MATNR
    GROUP BY s.VBELN, s.MATNR
    HAVING COUNT(*) > 1
    ORDER BY join_row_count DESC
    LIMIT 10
    """
)
print(f"Sales rows with multiple material matches: {len(result)}")
display(result if len(result) > 0 else "Clean join — no duplicates.")

---
## Daily Prompt

**Join sales orders to materials. Find all OPEN orders where the material has been discontinued (zero stock, no movement in 180 days) — these are fulfillment risk orders.**

Return: `VBELN`, `MATNR`, `MAKTX`, `NETWR`, `STATUS`, `LABST`, `RISK_REASON`

```sql
-- YOUR SOLUTION
SELECT
    s.VBELN,
    s.MATNR,
    m.MAKTX,
    s.NETWR,
    s.STATUS,
    m.LABST,
    'Discontinued material' AS RISK_REASON
FROM SALES s
INNER JOIN MATERIALS m ON s.MATNR = m.MATNR
WHERE
    s.STATUS = 'OPEN'
    AND m.LABST = 0
    -- YOUR SOLUTION: add condition for >180 days no movement
ORDER BY s.NETWR DESC
```

> **Hint:** SQLite date arithmetic: `julianday('now') - julianday(m.LAST_MOVEMENT_DATE) > 180`
> Combine zero stock AND no recent movement with AND.